# Baseline: Implementación y Comparación de Modelos

En esta sección del notebook, estableceremos nuestro modelo base (baseline) para la clasificación y predicción del riesgo de agotamiento académico utilizando el conjunto de datos `Salud_mental_agotamiento_estudiantil`. Basándonos en la evidencia de los artículos de referencia, hemos seleccionado **XGBoost (Extreme Gradient Boosting)** como nuestro algoritmo principal para esta implementación inicial por los siguientes motivos:

* **Rendimiento predictivo superior en la literatura:** Diversos estudios respaldan la alta eficacia de XGBoost en la detección del burnout. En la predicción de perfiles de riesgo, este algoritmo demostró ser el mejor modelo al alcanzar una exactitud (accuracy) del 97.53%, un recall del 99.99% y un F1-score del 94.73%.
* **Capacidad para manejar variables y relaciones complejas:**  El uso de machine learning, y específicamente de XGBoost, permite la detección de relaciones no lineales e interacciones complejas entre estas variables (como la regulación emocional y el sueño), un nivel de análisis difícil de capturar con modelos lineales o estadísticos tradicionales. Los algoritmos de *gradient boosting* en general han demostrado superioridad al capturar interacciones no lineales entre indicadores fisiológicos y patrones de comportamiento[cite: 6].


In [11]:
# 1. IMPORTACIONES Y CONEXIÓN A GOOGLE DRIVE
!pip install gdown
import gdown
from google.colab import drive
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

df = pd.read_csv("dataset_burnout_limpio.csv")


In [12]:
# 2. PREPROCESAMIENTO DE VARIABLES

X = df.drop(columns=['riesgo_burnout'])
y = df['riesgo_burnout']
X_encoded = pd.get_dummies(X, drop_first=True)


In [13]:
# 3. DIVISIÓN DEL CONJUNTO DE DATOS
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.30, random_state=42, stratify=y)

# Calcular el factor de balanceo para scale_pos_weight
# ratio = casos_negativos / casos_positivos
num_negativos = (y_train == 0).sum()
num_positivos = (y_train == 1).sum()
ratio_balanceo = num_negativos / num_positivos

In [14]:
# 4CONFIGURACIÓN DEL MODELO XGBOOST
#hiperpárametros proporcionados por el paper
xgb_model_balanced = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.3,
    max_depth=6,
    objective='binary:logistic',
    scale_pos_weight=ratio_balanceo, # Ajuste de peso para la clase minoritaria
    random_state=42
)

In [15]:
# 5. ENTRENAMIENTO Y EVALUACIÓN
xgb_model_balanced.fit(X_train, y_train)
y_pred_bal = xgb_model_balanced.predict(X_test)

# Calcular métricas
accuracy_bal = accuracy_score(y_test, y_pred_bal)
precision_bal = precision_score(y_test, y_pred_bal)
recall_bal = recall_score(y_test, y_pred_bal)
f1_bal = f1_score(y_test, y_pred_bal)

print("\n=== Resultados del Modelo Balanceado (scale_pos_weight) ===")
print(f"Exactitud (Accuracy): {accuracy_bal:.4f}")
print(f"Precisión (Precision): {precision_bal:.4f}")
print(f"Sensibilidad (Recall): {recall_bal:.4f}")
print(f"F1-Score: {f1_bal:.4f}")
print("\nReporte de Clasificación Detallado:")
print(classification_report(y_test, y_pred_bal))



=== Resultados del Modelo Balanceado (scale_pos_weight) ===
Exactitud (Accuracy): 0.8601
Precisión (Precision): 0.4424
Sensibilidad (Recall): 0.8937
F1-Score: 0.5918

Reporte de Clasificación Detallado:
              precision    recall  f1-score   support

           0       0.98      0.86      0.92    265949
           1       0.44      0.89      0.59     34051

    accuracy                           0.86    300000
   macro avg       0.71      0.87      0.75    300000
weighted avg       0.92      0.86      0.88    300000



El modelo base es XGBoost, el cual usa hiperparámetros del paper de Chowdhury et al., logró un F1-Score de 0.5924 debido al fuerte desbalance de clases.
En la siguiente etapa del desarrollo, implementaremos directamente técnicas de remuestreo para balancear las clases, probando Oversampling, Undersampling y SMOTE.
Además, modificaremos el modelo aplicando Class Weights y Upweighting para penalizar fuertemente los errores cometidos en la clase mayoritaria.
Con todas estas correcciones iterativas, buscaremos mejorar el desempeño del modelo antes de llegar a la solución final.